# 12 — Domain Presets & Configuration: From Prototype to Production

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/12_domain_presets_and_config.ipynb)

Director-AI ships 8 domain profiles with pre-tuned thresholds.
Configuration can come from code, environment variables, YAML files,
or profiles — and they compose cleanly.

**What you'll learn:**
1. Built-in domain profiles and their rationale
2. Configuration hierarchy (code → env → YAML → profile)
3. Scorer backends: heuristic, NLI, ONNX, hybrid, lite, Rust
4. Multi-GPU NLI sharding
5. LLM-as-judge escalation
6. Strict mode for safety-critical deployments

In [ ]:
!pip install -q director-ai

## 1. Domain Profiles

Each profile tunes thresholds for a specific use case.

In [ ]:
from director_ai.core.config import DirectorConfig

profiles = [
    "fast",
    "thorough",
    "research",
    "medical",
    "finance",
    "legal",
    "creative",
    "customer_support",
]

header = f"{'Profile':<18} {'Threshold':>9} {'Hard':>6} {'NLI':>5} {'w_logic':>7} {'w_fact':>7} {'Backend':>10}"
print(header)
print("─" * len(header))

for name in profiles:
    cfg = DirectorConfig.from_profile(name)
    print(
        f"{name:<18} {cfg.coherence_threshold:>9.2f} {cfg.hard_limit:>6.2f}"
        f" {cfg.use_nli!s:>5} {cfg.w_logic:>7.1f} {cfg.w_fact:>7.1f}"
        f" {cfg.scorer_backend:>10}",
    )

### Profile Rationale

| Profile | Why these settings |
|---------|-------------------|
| **fast** | Low threshold, no NLI — for latency-sensitive chatbots |
| **thorough** | High threshold, NLI enabled — for QA pipelines |
| **research** | Highest threshold — for scientific claim verification |
| **medical** | NLI required, high w_logic — patient safety |
| **finance** | High threshold — regulatory compliance |
| **legal** | Highest threshold — contract analysis |
| **creative** | Low threshold — fiction, marketing, brainstorming |
| **customer_support** | Balanced — accurate but not over-sensitive |

## 2. Configuration Hierarchy

Configuration sources, highest priority first:
1. **Code** — `DirectorConfig(threshold=0.7)`
2. **Environment variables** — `DIRECTOR_COHERENCE_THRESHOLD=0.7`
3. **YAML file** — `DirectorConfig.from_yaml("config.yaml")`
4. **Profile** — `DirectorConfig.from_profile("medical")`
5. **Defaults** — built-in defaults

In [ ]:
import os

# Method 1: Code
cfg = DirectorConfig(coherence_threshold=0.7, use_nli=True)
print(f"Code:    threshold={cfg.coherence_threshold}, nli={cfg.use_nli}")

# Method 2: Environment variables
os.environ["DIRECTOR_COHERENCE_THRESHOLD"] = "0.8"
os.environ["DIRECTOR_USE_NLI"] = "true"
cfg_env = DirectorConfig.from_env()
print(f"Env:     threshold={cfg_env.coherence_threshold}, nli={cfg_env.use_nli}")

# Method 3: Profile with overrides
cfg_med = DirectorConfig.from_profile("medical")
cfg_med.coherence_threshold = 0.85  # override profile default
print(f"Profile: threshold={cfg_med.coherence_threshold}, nli={cfg_med.use_nli}")

# Clean up
os.environ.pop("DIRECTOR_COHERENCE_THRESHOLD", None)
os.environ.pop("DIRECTOR_USE_NLI", None)

## 3. Scorer Backends

Director-AI supports 6 scoring backends via a plugin registry.

| Backend | Install | Latency | Accuracy | GPU |
|---------|---------|---------|----------|-----|
| `deberta` | `[nli]` | ~50ms | Highest | Optional |
| `onnx` | `[onnx]` | ~15ms | High | No |
| `minicheck` | `[minicheck]` | ~30ms | High | Optional |
| `lite` | (built-in) | <1ms | Moderate | No |
| `rust` | `[rust]` | <1ms | Moderate | No |
| `hybrid` | varies | ~20ms | Highest | Optional |

In [ ]:
from director_ai.core import list_backends

print("Registered scorer backends:")
for name, cls in sorted(list_backends().items()):
    print(f"  {name:<12} → {cls.__module__}.{cls.__qualname__}")

In [ ]:
# LiteScorer — zero-dependency heuristic scoring
from director_ai.core import LiteScorer

lite = LiteScorer()

# Test against ground truth
tests = [
    ("Paris is the capital of France.", "Paris", True),
    ("Berlin is the capital of France.", "Paris", False),
    ("The sky is blue due to scattering.", "blue", True),
    ("The sky is green.", "blue", False),
]

for response, fact, expected_pass in tests:
    score = lite.score(response, fact)
    status = "PASS" if score < 0.5 else "FAIL"
    match = "✓" if (score < 0.5) == expected_pass else "✗"
    print(f"  {match} [{status}] {score:.3f}  {response[:50]}")


## 4. Strict Mode

For safety-critical deployments, `strict_mode=True` disables heuristic
fallbacks. If the NLI model isn't available, scoring returns 0.9
(near-certain reject) rather than falling back to keyword matching.

In [ ]:
from director_ai.core import CoherenceScorer, GroundTruthStore

store = GroundTruthStore()
store.add("dosage", "Max ibuprofen: 400mg every 6 hours.")

# Normal mode: falls back to heuristics if NLI unavailable
scorer_normal = CoherenceScorer(
    threshold=0.5,
    ground_truth_store=store,
    strict_mode=False,
    use_nli=False,
)
approved, score = scorer_normal.review("dosage", "Take 400mg every 6 hours.")
print(f"Normal mode: approved={approved}, score={score.score:.3f}")

# Strict mode: rejects when NLI not available
scorer_strict = CoherenceScorer(
    threshold=0.5,
    ground_truth_store=store,
    strict_mode=True,
    use_nli=False,
)
approved, score = scorer_strict.review("dosage", "Take 400mg every 6 hours.")
print(f"Strict mode: approved={approved}, score={score.score:.3f}")
if hasattr(score, "strict_mode_rejected"):
    print(f"  strict_mode_rejected={score.strict_mode_rejected}")

## 5. Multi-GPU NLI Sharding

For high-throughput production, shard NLI inference across GPUs.

```python
from director_ai.core import ShardedNLIScorer

scorer = ShardedNLIScorer(devices=["cuda:0", "cuda:1", "cuda:2"])
# Automatically round-robins inference across GPUs
```

Or via config:
```yaml
nli_devices: "cuda:0,cuda:1"
```

## 6. LLM-as-Judge Escalation

When NLI confidence is below a threshold, escalate to an LLM judge
for a second opinion.

```python
config = DirectorConfig(
    llm_judge_enabled=True,
    llm_judge_confidence_threshold=0.3,  # escalate if NLI < 0.3
    llm_judge_provider="openai",
    llm_judge_model="gpt-4o-mini",
)
```

The final score blends NLI (70%) and LLM judge (30%) when escalation
triggers. This catches edge cases where the NLI model is uncertain.

## Environment Variable Reference

All config fields can be set via `DIRECTOR_` prefixed env vars:

| Env Var | Default | Description |
|---------|---------|-------------|
| `DIRECTOR_COHERENCE_THRESHOLD` | 0.6 | Minimum coherence |
| `DIRECTOR_HARD_LIMIT` | 0.5 | Emergency stop floor |
| `DIRECTOR_USE_NLI` | false | Enable DeBERTa |
| `DIRECTOR_NLI_MODEL` | FactCG-DeBERTa-v3-Large | NLI model |
| `DIRECTOR_SCORER_BACKEND` | deberta | Backend type |
| `DIRECTOR_NLI_DEVICES` | (empty) | Multi-GPU devices |
| `DIRECTOR_VECTOR_BACKEND` | memory | Vector store |
| `DIRECTOR_LOG_LEVEL` | WARNING | Logging level |
| `DIRECTOR_LOG_JSON` | false | JSON logging |
| `DIRECTOR_LLM_JUDGE_ENABLED` | false | LLM escalation |